In [ ]:
from pathlib import Path
BASE = Path("/home/lka/HYBRID_SARIMA")
print(BASE.exists(), BASE)

#Import the file to read it
import pandas as pd
disocupati_trimestral = pd.read_csv(BASE / 'data.csv', 
                       index_col='Period', 
                       parse_dates=True)
print(disocupati_trimestral)

In [ ]:
#Dividir el dataset en training, validation y test
series = [
    '15-34 M', '15-34 F',
    '35-49 M', '35-49 F',
    '50-74 M', '50-74 F',
    'Global_Rate'
]

# Ordenar cronológicamente
df = disocupati_trimestral.sort_index()

# División 70% Train, 15% Validation, 15% Test
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df[series].iloc[:train_end].copy()
val = df[series].iloc[train_end:val_end].copy()
test = df[series].iloc[val_end:].copy()

# Mostrar resultados para cada serie
#for serie in series:
#    print(f"\nSerie: {serie}")
#    print(f"Train: {len(train[serie])} observaciones")
#    print(f"Validation: {len(val[serie])} observaciones")
#    print(f"Test: {len(test[serie])} observaciones")

print(train)
print(val)
print(test)


In [ ]:

import warnings
warnings.filterwarnings("ignore")
import itertools,copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from torch.utils.data import DataLoader,TensorDataset

torch.manual_seed(42)
np.random.seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

class VanillaLSTM(nn.Module):
    def __init__(self,neurons,dropout):
        super().__init__()
        self.lstms=nn.ModuleList()
        input_size=1
        for n in neurons:
            self.lstms.append(nn.LSTM(input_size=input_size,hidden_size=n,num_layers=1,batch_first=True))
            input_size=n
        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(neurons[-1],1)

    def forward(self,x):
        for i,lstm in enumerate(self.lstms):
            x,_=lstm(x)
            if i<len(self.lstms)-1:
                x=self.dropout(x)
        if len(self.lstms)==1:
            x=self.dropout(x[:,-1,:])
        else:
            x=x[:,-1,:]
        return self.fc(x)
def create_sequences(data,window,start,end):
    X=np.array([data[i-window:i] for i in range(start,end)],dtype=np.float32)
    y=np.array([data[i] for i in range(start,end)],dtype=np.float32)
    return X.reshape(-1,window,1),y.reshape(-1,1)

neuron_options=[32,64,128]
architectures=[(n,) for n in neuron_options]+[(n1,n2) for n1 in neuron_options for n2 in neuron_options if n1>=n2]
# Define the hyperparameter search space for LSTM models
# learning rates to try
learning_rates=[0.01,0.001]
# window sizes to try
window_sizes=[4,6,8]
# epochs to try
epochs_options=[100,200]
# batch sizes to try
batch_sizes=[16,32]
# dropout rates to try
dropout_options=[0.1,0.3,0.5]

params=list(itertools.product(architectures,learning_rates,window_sizes,epochs_options,batch_sizes,dropout_options))
patience,min_delta=15,1e-5
lstm_models,results={},[]

for col in train.columns:
    y_train=train[col].dropna().to_numpy(dtype=np.float32)
    y_val=val[col].dropna().to_numpy(dtype=np.float32)
    candidatos=[]
    if len(y_train)<=8 or len(y_val)==0:
        lstm_models[col]=[]
        print(f"{col} (sin datos suficientes)")
        continue
    scaler=MinMaxScaler()
    train_scaled=scaler.fit_transform(y_train.reshape(-1,1)).flatten()
    val_scaled=scaler.transform(y_val.reshape(-1,1)).flatten()
    data=np.concatenate([train_scaled,val_scaled])
    for neurons,lr,window,epochs,batch_size,dropout in params:
        if len(y_train)<=window:
            continue
        torch.manual_seed(42)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(42)
        layers=len(neurons)
        X_train,Y_train=create_sequences(data,window,window,len(y_train))
        X_val,Y_val=create_sequences(data,window,len(y_train),len(data))
        loader=DataLoader(TensorDataset(torch.tensor(X_train),torch.tensor(Y_train)),batch_size=batch_size,shuffle=False)
        X_val_t=torch.tensor(X_val).to(device)
        Y_val_t=torch.tensor(Y_val).to(device)
        model=VanillaLSTM(neurons,dropout).to(device)
        optimizer=torch.optim.Adam(model.parameters(),lr=lr)
        criterion=nn.MSELoss()
        best_val_loss=float("inf")
        best_state=None
        best_epoch=counter=epochs_trained=0
        for epoch in range(epochs):
            model.train()
            for xb,yb in loader:
                xb,yb=xb.to(device),yb.to(device)
                optimizer.zero_grad()
                loss=criterion(model(xb),yb)
                loss.backward()
                optimizer.step()
            model.eval()
            with torch.no_grad():
                val_loss=criterion(model(X_val_t),Y_val_t).item()
            epochs_trained=epoch+1
            if not np.isfinite(val_loss):
                break
            if val_loss<best_val_loss-min_delta:
                best_val_loss=val_loss
                best_state=copy.deepcopy(model.state_dict())
                best_epoch=epoch+1
                counter=0
            else:
                counter+=1
            if counter>=patience:
                break
        if best_state is None:
            del model
            continue
        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            pred_scaled=model(X_val_t).cpu().numpy()
            fitted_scaled=model(torch.tensor(X_train).to(device)).cpu().numpy()
        pred=scaler.inverse_transform(pred_scaled).flatten()
        fitted=scaler.inverse_transform(fitted_scaled).flatten()
        real=y_val
        values=y_train
        if not np.all(np.isfinite(pred)) or not np.all(np.isfinite(fitted)):
            del model
            continue
        mae=mean_absolute_error(real,pred)
        mse=mean_squared_error(real,pred)
        rmse=np.sqrt(mse)
        mask=real!=0
        mape=np.mean(np.abs((real[mask]-pred[mask])/real[mask]))*100 if mask.any() else np.nan
        scale=np.mean(np.abs(values[4:]-values[:-4])) if len(values)>4 else np.nan
        mase=mae/scale if np.isfinite(scale) and scale>0 else np.nan
        rmsee=np.sqrt(np.mean((values[window:]-fitted)**2))
        r2=r2_score(real,pred) if len(real)>1 else np.nan
        candidatos.append({
            "Serie":col,"Layers":layers,"Neurons":"-".join(map(str,neurons)),
            "Optimizer":"Adam","Learning_Rate":lr,"Window":window,
            "Epochs":epochs,"Best_Epoch":best_epoch,"Epochs_Trained":epochs_trained,
            "Batch_Size":batch_size,"Dropout":dropout,"MAE":mae,"MSE":mse,
            "RMSE":rmse,"MAPE":mape,"MASE":mase,"RMSEE":rmsee,"R2":r2
        })
        del model
    candidatos=sorted(candidatos,key=lambda x:x["RMSE"])
    lstm_models[col]=candidatos[:10]
    for rank,result in enumerate(candidatos[:10],start=1):
        results.append({"Serie":col,"Rank":rank,**{k:v for k,v in result.items() if k!="Serie"}})
    print(f"{col} (ok)")

top10_lstm=pd.DataFrame(results).round(4)
BASE=Path(BASE) if "BASE" in globals() else Path(".")
(BASE/"results"/"LSTM").mkdir(parents=True,exist_ok=True)
top10_lstm.to_csv(BASE/"results"/"LSTM"/"top10_lstm.csv",index=False)
display(top10_lstm)
